# AI-Lake + DuckDB

DuckDB reads AI-Lake `.parquet` files as standard Parquet — no plugin required.
The HNSW footer bytes are invisible to DuckDB; it reads tabular columns normally.

**What this notebook covers:**
1. Direct Parquet scan (glob)
2. Filtered queries + aggregations
3. `embedding` column as `BLOB`
4. Iceberg extension scan (optional)
5. Performance: row count, column stats

In [ ]:
import os
import pathlib
import duckdb
import pandas as pd

TABLE_PATH = os.environ.get('DEMO_TABLE_PATH', '/data/ailake_demo')
PARQUET_GLOB = str(pathlib.Path(TABLE_PATH) / '**' / '*.parquet')

con = duckdb.connect()
print(f'DuckDB {duckdb.__version__}')
print(f'Parquet glob: {PARQUET_GLOB}')

## 1. Basic scan

In [ ]:
total = con.execute(f"""
    SELECT count(*) AS rows
    FROM read_parquet('{PARQUET_GLOB}', hive_partitioning=false)
""").fetchone()[0]
print(f'Total rows: {total}')

In [ ]:
# Schema: DuckDB exposes the embedding column as BLOB (raw F16 bytes)
con.execute(f"""
    DESCRIBE SELECT * FROM read_parquet('{PARQUET_GLOB}', hive_partitioning=false) LIMIT 1
""").df()

In [ ]:
con.execute(f"""
    SELECT text, octet_length(embedding) AS embedding_bytes
    FROM read_parquet('{PARQUET_GLOB}', hive_partitioning=false)
    LIMIT 5
""").df()

## 2. Filtered queries

In [ ]:
# Full-text keyword filter (pushed down to Parquet column scan)
df = con.execute(f"""
    SELECT text
    FROM read_parquet('{PARQUET_GLOB}', hive_partitioning=false)
    WHERE text LIKE '%vector search%'
""").df()
print(f"Rows mentioning 'vector search': {len(df)}")
df.head(3)

## 3. Aggregations

In [ ]:
con.execute(f"""
    SELECT
        count(*)                    AS total_rows,
        avg(length(text))           AS avg_text_len,
        min(length(text))           AS min_text_len,
        max(length(text))           AS max_text_len,
        avg(octet_length(embedding))      AS avg_emb_bytes,
        sum(octet_length(embedding))      AS total_emb_bytes
    FROM read_parquet('{PARQUET_GLOB}', hive_partitioning=false)
""").df()

## 4. Topic distribution

In [ ]:
# Extract topic keyword (text was generated as "An intro to <topic>: ...")
topics = [
    'machine learning', 'database systems', 'vector search', 'data lakes',
    'Apache Iceberg', 'embedding models', 'RAG pipelines', 'LLM inference',
    'semantic search', 'quantization',
]
rows = []
for t in topics:
    n = con.execute(f"""
        SELECT count(*) FROM read_parquet('{PARQUET_GLOB}', hive_partitioning=false)
        WHERE text LIKE '%{t}%'
    """).fetchone()[0]
    rows.append({'topic': t, 'count': n})
pd.DataFrame(rows).sort_values('count', ascending=False)

## 5. Iceberg extension (optional)

DuckDB's Iceberg extension reads the Iceberg snapshot metadata and discovers
the Parquet files from the manifest — the same as a standard Iceberg catalog read.

In [ ]:
import json

# Resolve latest snapshot metadata path
meta_dir = pathlib.Path(TABLE_PATH) / 'default' / 'table' / 'metadata'
hint = (meta_dir / 'version-hint.text').read_text().strip()
meta_path = meta_dir / f'v{hint}.metadata.json'
meta = json.loads(meta_path.read_text())
print(f'Table UUID    : {meta["table-uuid"]}')
print(f'Format version: {meta["format-version"]}')
print(f'Snapshots     : {len(meta.get("snapshots", []))}')
print()

try:
    con.execute('INSTALL iceberg; LOAD iceberg;')
    iceberg_tbl = str(pathlib.Path(TABLE_PATH) / 'default' / 'table')
    n = con.execute(f"SELECT count(*) FROM iceberg_scan('{iceberg_tbl}')").fetchone()[0]
    print(f'Iceberg extension scan: {n} rows')
except Exception as exc:
    print(f'Iceberg extension unavailable ({exc}) — using Parquet glob scan instead')

## 6. Per-file storage breakdown

Each AI-Lake `.parquet` is self-contained: Parquet data + HNSW footer.
DuckDB reports file sizes; the last ~`hnsw_size` bytes are the index.


In [ ]:
import os

parquet_files = sorted(pathlib.Path(TABLE_PATH).rglob('*.parquet'))
rows = []
for p in parquet_files:
    size = p.stat().st_size
    count = con.execute(
        f"SELECT count(*) FROM read_parquet('{p}', hive_partitioning=false)"
    ).fetchone()[0]
    rows.append({'file': p.name, 'size_bytes': size, 'rows': count,
                 'bytes_per_row': size // max(count, 1)})

import pandas as pd
pd.DataFrame(rows)


## 7. Decode F16 embedding from DuckDB BLOB

The `embedding` column is stored as `FIXED_LEN_BYTE_ARRAY` (F16).
DuckDB exposes it as `BLOB`. Decode back to float32 in Python with `numpy`.


In [ ]:
import numpy as np

# Fetch the raw embedding bytes for row 0
row = con.execute(
    f"SELECT text, embedding FROM read_parquet('{PARQUET_GLOB}', hive_partitioning=false) LIMIT 1"
).fetchone()
text_val, emb_blob = row

# Decode F16 bytes → float32
emb_f16 = np.frombuffer(bytes(emb_blob), dtype=np.float16)
emb_f32 = emb_f16.astype(np.float32)

print(f'Text            : {text_val[:60]}...')
print(f'Embedding bytes : {len(emb_blob)} (= dim × 2 for F16)')
print(f'Decoded dim     : {len(emb_f32)}')
print(f'First 8 values  : {emb_f32[:8].round(4)}')


## 8. Iceberg metadata — snapshot and schema via JSON

Read the Iceberg `metadata.json` directly to inspect table properties
and AI-Lake custom metadata without any Iceberg library.


In [ ]:
import json

meta_dir  = pathlib.Path(TABLE_PATH) / 'default' / 'table' / 'metadata'
hint      = (meta_dir / 'version-hint.text').read_text().strip()
meta_path = meta_dir / f'v{hint}.metadata.json'
meta      = json.loads(meta_path.read_text())

print(f'Table UUID      : {meta["table-uuid"]}')
print(f'Format version  : {meta["format-version"]}')
print(f'Snapshots       : {len(meta.get("snapshots", []))}')
print()
print('AI-Lake properties:')
for k, v in sorted(meta.get('properties', {}).items()):
    if k.startswith('ailake.'):
        print(f'  {k:40s} = {v}')


## Key takeaway

DuckDB reads AI-Lake tables as **standard Parquet** — zero configuration, zero plugins.
The HNSW index lives in the file footer after the final `PAR1` magic bytes;
DuckDB stops at `PAR1` and never touches the index section.

The `embedding` column has `octet_length(embedding) = dim * 2` bytes (F16 storage, 2 bytes/dimension).
With dim=32 the demo fixture stores 64 bytes per vector.

For vector similarity search, use `ailake.search()` — see `01_ailake_demo.ipynb`.